In [ ]:
import torch.nn as nn
from torchvision import transforms, models
import pandas as pd
import pydicom
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm # Para a barra de progresso

In [ ]:
# Carrega a ResNet50 pré-treinada no ImageNet
model = models.resnet50(weights='IMAGENET1K_V1')

# Ajusta a camada de entrada (Mamografias são escala de cinza, ImageNet é RGB)
# Opção A: Converter imagem para 3 canais (mais simples)
# Opção B: Alterar a primeira camada para 1 canal:
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Ajusta a última camada para o seu número de classes (ex: BI-RADS 1 a 5 = 5 classes)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)

In [ ]:
class VinDrMLODataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        df = pd.read_csv(csv_file)
        self.data = df[df['view'] == 'MLO'].reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        # NOVO MAPEAMENTO BINÁRIO:
        # BI-RADS 1, 2, 3 -> Classe 0 (Benigno)
        # BI-RADS 4, 5    -> Classe 1 (Suspeito/Maligno)
        self.label_map = {1: 0, 2: 0, 3: 0, 4: 1, 5: 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Monta o caminho: root/study_id/image_id.dicom
        img_path = f"{self.root_dir}/{row['study_id']}/{row['image_id']}.dicom"
        
        # Leitura do DICOM
        ds = pydicom.dcmread(img_path)
        pixel_array = ds.pixel_array.astype(float)
        
        # Normalização Básica (Min-Max)
        pixel_array = (pixel_array - np.min(pixel_array)) / (np.max(pixel_array) - np.min(pixel_array))
        pixel_array = (pixel_array * 255).astype(np.uint8)
        
        image = Image.fromarray(pixel_array).convert('RGB') # ResNet espera 3 canais por padrão
        
        label = self.label_map[row['breast_birads']]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# Augmentation focado em mamografia
train_transform = transforms.Compose([
    transforms.Resize((512, 512)), # Resolução maior ajuda em lesões pequenas
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Inicializando o Modelo (mantendo as imagens em RGB no Dataset)
model = models.resnet50(weights='IMAGENET1K_V1')

# Ajusta a última camada para 2 classes (Benigno vs Maligno)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Exemplo de Class Weights
# Pesos inversamente proporcionais à frequência das classes no seu CSV
# Como os casos malignos (Classe 1) são muito mais raros que os benignos (Classe 0),
# precisamos dar um peso muito maior para a rede "prestar atenção" neles.
# O ideal é: (Total de amostras Classe 0) / (Total de amostras Classe 1)
# Supondo que 90% seja benigno e 10% maligno, o peso ficaria 1 para Benigno e 9 para Maligno.

# Exemplo de Pesos para o caso binário:
weights = torch.tensor([1.0, 8.0]).to(device) # Ajuste o 8.0 após contar no seu CSV

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # Um learning rate baixo é bom para Transfer Learning

In [ ]:
# Exemplo de criação dos DataLoaders
# train_dataset = VinDrMLODataset('train_labels.csv', root_dir='...', transform=train_transform)
# val_dataset = VinDrMLODataset('val_labels.csv', root_dir='...', transform=val_transform)

BATCH_SIZE = 16 # Ajuste conforme a memória da sua GPU (Tente 8 ou 16 para ResNet50 em 512x512)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
num_epochs = 20
best_auc = 0.0 # Para salvarmos o melhor modelo

for epoch in range(num_epochs):
    print(f"\n--- Época {epoch+1}/{num_epochs} ---")
    
    # ==========================
    #       TREINAMENTO
    # ==========================
    model.train()
    train_loss = 0.0
    
    # tqdm cria uma barra de progresso
    loop_treino = tqdm(train_loader, desc="Treino", leave=False)
    
    for images, labels in loop_treino:
        images, labels = images.to(device), labels.to(device)
        
        # 1. Zerar os gradientes
        optimizer.zero_grad()
        
        # 2. Forward pass (Previsões)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 3. Backward pass (Calcular gradientes)
        loss.backward()
        
        # 4. Otimizar (Atualizar pesos)
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        loop_treino.set_postfix(loss=loss.item())
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # ==========================
    #        VALIDAÇÃO
    # ==========================
    model.eval()
    val_loss = 0.0
    
    # Listas para guardar as previsões e os rótulos reais para o Scikit-Learn
    all_preds = []
    all_labels = []
    all_probs = [] # Necessário para o AUC-ROC
    
    with torch.no_grad(): # Desliga o cálculo de gradientes para poupar memória e tempo
        loop_val = tqdm(val_loader, desc="Validação", leave=False)
        for images, labels in loop_val:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            # Pega as probabilidades da classe 1 (Maligno) usando Softmax
            probs = F.softmax(outputs, dim=1)[:, 1]
            
            # Pega a classe com maior probabilidade (0 ou 1)
            _, preds = torch.max(outputs, 1)
            
            # Salva na lista
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # ==========================
    #    CÁLCULO DE MÉTRICAS
    # ==========================
    # A Matriz de Confusão nos dá: Verdadeiros Negativos, Falsos Positivos, Falsos Negativos, Verdadeiros Positivos
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    
    # Sensibilidade (Recall): Dos que eram câncer, quantos acertamos?
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Especificidade: Dos que eram benignos, quantos acertamos?
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # AUC - Area Under the ROC Curve
    auc = roc_auc_score(all_labels, all_probs)
    
    print(f"Loss Treino: {train_loss:.4f} | Loss Validação: {val_loss:.4f}")
    print(f"Sensibilidade (Recall): {sensitivity:.4f}")
    print(f"Especificidade:       {specificity:.4f}")
    print(f"AUC-ROC:              {auc:.4f}")
    
    # ==========================
    #  SALVAR O MELHOR MODELO
    # ==========================
    # Em medicina, geralmente salvamos o modelo com o melhor AUC ou a melhor Sensibilidade
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'melhor_modelo_vindr_mlo.pth')
        print(f"🔥 Novo melhor modelo salvo! (AUC: {best_auc:.4f})")